
# Enhanced Visual Analysis Notebook

This notebook improves styling, adds robust exploratory analysis, and **saves every visual** to `./charts`.
It is designed to be **generic** and work with the data and calculations produced by your existing notebook.


In [ ]:

# --- Configuration & Imports ---
import os, re, itertools, math
from typing import Optional, List
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Keep visuals clean and generic (no explicit styles/colors)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True

CHARTS_DIR = './charts'
os.makedirs(CHARTS_DIR, exist_ok=True)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')


In [ ]:

# --- Utility & Plot Helpers ---
def _sanitize_name(name: str) -> str:
    import re
    if not isinstance(name, str):
        name = str(name)
    name = name.strip().replace(' ', '_')
    name = re.sub(r'[^0-9a-zA-Z_\-]+', '', name)
    return name or 'unnamed'

def save_figure(filename: str):
    fn = f"{CHARTS_DIR}/{_sanitize_name(filename)}.png"
    plt.tight_layout()
    plt.savefig(fn, dpi=150, bbox_inches='tight')
    plt.close()
    return fn

def detect_datetime_column(df: pd.DataFrame) -> Optional[str]:
    dt_cols = [c for c in df.columns if np.issubdtype(df[c].dtype, np.datetime64)]
    if dt_cols:
        return dt_cols[0]
    for c in df.columns:
        if c.lower() in ['date','datetime','timestamp','time','asof_date']:
            try:
                pd.to_datetime(df[c])
                return c
            except Exception:
                pass
    return None

def numeric_columns(df: pd.DataFrame) -> List[str]:
    return df.select_dtypes(include=[np.number]).columns.tolist()

def basic_summary(df: pd.DataFrame) -> pd.DataFrame:
    num_cols = numeric_columns(df)
    if not num_cols:
        return pd.DataFrame()
    desc = df[num_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
    desc['skew'] = df[num_cols].skew(numeric_only=True)
    desc['kurtosis'] = df[num_cols].kurtosis(numeric_only=True)
    return desc

def plot_missingness(df: pd.DataFrame, name: str):
    miss = df.isna().sum()
    if miss.sum() == 0:
        return
    plt.figure()
    miss.plot(kind='bar')
    plt.title(f"Missing values per column: {name}")
    plt.xlabel('Column'); plt.ylabel('Count')
    save_figure(f"{name}__missingness")

def plot_histograms(df: pd.DataFrame, name: str, bins: int=30, max_cols: int=30):
    num_cols = numeric_columns(df)
    for col in num_cols[:max_cols]:
        plt.figure()
        df[col].dropna().plot(kind='hist', bins=bins, alpha=0.8)
        plt.title(f"Histogram: {name}.{col}")
        plt.xlabel(col); plt.ylabel('Frequency')
        save_figure(f"{name}__hist__{col}")

def plot_boxplots(df: pd.DataFrame, name: str, max_cols: int=30):
    num_cols = numeric_columns(df)
    for col in num_cols[:max_cols]:
        plt.figure()
        plt.boxplot(df[col].dropna().values, vert=True)
        plt.title(f"Boxplot: {name}.{col}"); plt.ylabel(col)
        save_figure(f"{name}__box__{col}")

def plot_correlation_heatmap(df: pd.DataFrame, name: str):
    cols = numeric_columns(df)
    if len(cols) < 2: return
    corr = df[cols].corr()
    plt.figure()
    plt.imshow(corr.values, aspect='auto', interpolation='nearest')
    plt.xticks(range(len(cols)), cols, rotation=90)
    plt.yticks(range(len(cols)), cols)
    plt.title(f"Correlation heatmap: {name}")
    plt.colorbar()
    save_figure(f"{name}__correlation_heatmap")

def plot_pairwise_scatter(df: pd.DataFrame, name: str, max_pairs: int=10):
    cols = numeric_columns(df)
    if len(cols) < 2: return
    for a, b in list(itertools.combinations(cols, 2))[:max_pairs]:
        plt.figure()
        plt.scatter(df[a], df[b], s=8)
        plt.title(f"Scatter: {name}.{a} vs {b}")
        plt.xlabel(a); plt.ylabel(b)
        save_figure(f"{name}__scatter__{a}__{b}")

def plot_time_series(df: pd.DataFrame, name: str, date_col: Optional[str]):
    cols = numeric_columns(df)
    if not cols: return
    if date_col and date_col in df.columns:
        x = pd.to_datetime(df[date_col], errors='coerce')
        ok = x.notna(); df = df.loc[ok].sort_values(date_col); x = x.loc[ok]
        for col in cols:
            plt.figure(); plt.plot(x, df[col].values)
            plt.title(f"Time series: {name}.{col}"); plt.xlabel(date_col); plt.ylabel(col)
            save_figure(f"{name}__ts__{col}")
    elif isinstance(df.index, pd.DatetimeIndex):
        for col in cols:
            plt.figure(); plt.plot(df.index, df[col].values)
            plt.title(f"Time series: {name}.{col}"); plt.xlabel('date'); plt.ylabel(col)
            save_figure(f"{name}__ts__{col}")

def plot_rolling_stats(df: pd.DataFrame, name: str, date_col: Optional[str], windows=(5, 20, 60)):
    cols = numeric_columns(df)
    if not cols: return
    if date_col and date_col in df.columns:
        x = pd.to_datetime(df[date_col], errors='coerce')
        ok = x.notna(); df = df.loc[ok].sort_values(date_col); x = x.loc[ok]
    elif isinstance(df.index, pd.DatetimeIndex):
        x = df.index
    else:
        x = None
    for col in cols:
        for w in windows:
            if len(df[col].dropna()) < max(10, w + 5): continue
            rm = df[col].rolling(w).mean(); rs = df[col].rolling(w).std()
            xi = x if x is not None else df.index
            plt.figure(); plt.plot(xi, rm, label=f'mean({w})'); plt.plot(xi, rs, label=f'std({w})')
            plt.title(f"Rolling mean/std (w={w}): {name}.{col}")
            plt.xlabel('date' if x is not None else 'index'); plt.ylabel(col)
            plt.legend(); save_figure(f"{name}__rolling_w{w}__{col}")

def plot_autocorrelation(df: pd.DataFrame, name: str, max_lag: int=20):
    cols = numeric_columns(df)
    for col in cols:
        x = df[col].dropna().values
        if len(x) < max_lag + 5: continue
        x = x - np.mean(x)
        denom = np.dot(x, x); acf = [1.0]
        for lag in range(1, max_lag + 1):
            acf.append(np.dot(x[:-lag], x[lag:]) / denom)
        plt.figure(); plt.bar(range(len(acf)), acf)
        plt.title(f"Autocorrelation (max_lag={max_lag}): {name}.{col}")
        plt.xlabel('lag'); plt.ylabel('acf')
        save_figure(f"{name}__acf__{col}")

def zscore_outliers(df: pd.DataFrame, name: str, threshold: float=3.0) -> pd.DataFrame:
    cols = numeric_columns(df)
    if not cols: return pd.DataFrame()
    zs = (df[cols] - df[cols].mean()) / df[cols].std(ddof=0)
    out = df[(np.abs(zs) > threshold).any(axis=1)].copy()
    return out.assign(__outlier=True)

def analyze_dataframe(df: pd.DataFrame, name: str):
    name = _sanitize_name(name or 'data')
    summary = basic_summary(df)
    if not summary.empty:
        display(summary)
        summary.to_csv(f"{CHARTS_DIR}/{name}__summary.csv")
    plot_missingness(df, name)
    plot_histograms(df, name)
    plot_boxplots(df, name)
    plot_correlation_heatmap(df, name)
    plot_pairwise_scatter(df, name)
    dt_col = detect_datetime_column(df)
    plot_time_series(df, name, dt_col)
    plot_rolling_stats(df, name, dt_col)
    plot_autocorrelation(df, name)
    out = zscore_outliers(df, name, threshold=3.0)
    if not out.empty:
        display(out.head(20))
        out.to_csv(f"{CHARTS_DIR}/{name}__zscore_outliers.csv", index=False)
    print(f"Analysis complete for DataFrame: {name}")



## Optional: Alpha/Beta Parameter Grid (Scaffolding)

Implement `evaluate(alpha, beta)` to return your metric and use the grid to visualize results.


In [ ]:

def evaluate(alpha: float, beta: float) -> float:
    raise NotImplementedError("Implement evaluate(alpha, beta) using your pipeline's metric.")

def param_grid_search(alphas, betas):
    vals = np.zeros((len(alphas), len(betas)))
    for i, a in enumerate(alphas):
        for j, b in enumerate(betas):
            vals[i, j] = evaluate(a, b)
    return vals

def plot_param_heatmap(alphas, betas, values, title='Parameter grid metric'):
    plt.figure()
    plt.imshow(values, aspect='auto', interpolation='nearest', origin='lower')
    plt.xticks(range(len(betas)), [f"{b:.3g}" for b in betas], rotation=90)
    plt.yticks(range(len(alphas)), [f"{a:.3g}" for a in alphas])
    plt.xlabel('beta'); plt.ylabel('alpha'); plt.title(title)
    plt.colorbar()
    save_figure('grid__alpha_beta_heatmap')

# Example (after you implement evaluate):
# alphas = np.linspace(0.0, 1.0, 21)
# betas  = np.linspace(0.0, 1.0, 21)
# grid = param_grid_search(alphas, betas)
# plot_param_heatmap(alphas, betas, grid, title='Metric by alpha/beta')



## Load Data from the Original Notebook

Set the path to your original notebook if needed.


In [ ]:

ORIGINAL_NOTEBOOK_PATHS = ['notebook.ipynb', './notebook.ipynb', '/mnt/data/notebook.ipynb']

_loaded = False
for _p in ORIGINAL_NOTEBOOK_PATHS:
    try:
        import IPython
        ip = IPython.get_ipython()
        if ip is None:
            raise RuntimeError("Not inside IPython.")
        ip.run_line_magic('run', _p)
        print(f"Loaded original notebook: {_p}")
        _loaded = True
        break
    except Exception as e:
        print(f"Tried {_p} -> {e}")

if not _loaded:
    print("WARNING: Could not run the original notebook. Adjust ORIGINAL_NOTEBOOK_PATHS and re-run.")



## Auto-Discover DataFrames and Run Analysis


In [ ]:

df_names = []
for k, v in sorted(globals().items()):
    try:
        if isinstance(v, pd.DataFrame) and len(v) > 0:
            df_names.append(k)
    except Exception:
        pass

print("Discovered DataFrames:", df_names)

for name in df_names:
    try:
        df = globals()[name]
        print(f"\n=== Analyzing: {name} (shape={df.shape}) ===")
        analyze_dataframe(df, name)
    except Exception as e:
        print(f"Skipping {name} due to error: {e}")



## Optional Targeted Time-Series Diagnostics


In [ ]:

PRIMARY_DF_NAME = None
DATE_COL = None
VALUE_COL = None

if PRIMARY_DF_NAME and DATE_COL and VALUE_COL:
    try:
        dfp = globals()[PRIMARY_DF_NAME].copy()
        dfp[DATE_COL] = pd.to_datetime(dfp[DATE_COL], errors='coerce')
        dfp = dfp.dropna(subset=[DATE_COL, VALUE_COL]).sort_values(DATE_COL)
        for w in [5, 20, 60]:
            if len(dfp) > w + 5:
                rm = dfp[VALUE_COL].rolling(w).mean()
                rs = dfp[VALUE_COL].rolling(w).std()
                plt.figure(); plt.plot(dfp[DATE_COL], rm, label=f'mean({w})'); plt.plot(dfp[DATE_COL], rs, label=f'std({w})')
                plt.title(f'Primary series rolling mean/std (w={w})')
                plt.xlabel(DATE_COL); plt.ylabel(VALUE_COL); plt.legend()
                save_figure(f'primary__rolling_w{w}__{VALUE_COL}')
        z = (dfp[VALUE_COL] - dfp[VALUE_COL].mean()) / dfp[VALUE_COL].std(ddof=0)
        dfp['__anomaly'] = np.abs(z) > 3.0
        if dfp['__anomaly'].any():
            anomalies = dfp.loc[dfp['__anomaly'], [DATE_COL, VALUE_COL]]
            anomalies.to_csv(f'{CHARTS_DIR}/primary__anomalies.csv', index=False)
            print('Saved anomaly rows for primary series -> primary__anomalies.csv')
    except Exception as e:
        print(f"Primary diagnostics skipped due to error: {e}")
